In [13]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/media_recommender/data', exist_ok=True)
!cp -r /content/drive/MyDrive/media_recommender_data/* /content/media_recommender/data/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# ALS — Alternating Least Squares

The main collaborative filtering model. Unlike item-item CF (which directly compares raw user-overlap between every pair of anime), ALS *learns* a compressed representation — a small vector of numbers per user and per anime — such that multiplying a user's vector by an anime's vector approximates fit. It's more scalable (no need to store a full N×N similarity matrix) and generally generalizes better on sparse data.

Reuses the sparse matrix built in `02_baseline.ipynb`.

In [20]:
from scipy.sparse import load_npz
import os
import pickle
import implicit
import pandas as pd
import kagglehub
import numpy as np
import shutil


In [15]:
DATA_DIR = '/content/media_recommender/data'

train = pd.read_parquet(os.path.join(DATA_DIR, "train_ratings.parquet"))
test = pd.read_parquet(os.path.join(DATA_DIR, "test_ratings.parquet"))
animes = pd.read_parquet(os.path.join(DATA_DIR, "animes.parquet"))

anime_ids = train['anime_id'].astype('category')
anime_id_map = dict(enumerate(anime_ids.cat.categories))
anime_id_map_reverse = {v: k for k, v in anime_id_map.items()}

user_ids = train['user_id'].astype('category')
user_id_map = dict(enumerate(user_ids.cat.categories))
user_id_map_reverse = {v: k for k, v in user_id_map.items()}

train['anime_idx'] = anime_ids.cat.codes
train['user_idx'] = user_ids.cat.codes
train['anime_idx'] = train['anime_idx'].astype(int)
train['user_idx'] = train['user_idx'].astype(int)

test['anime_idx'] = test['anime_id'].map(anime_id_map_reverse)
test['user_idx'] = test['user_id'].map(user_id_map_reverse)



## Train ALS

`item_user_matrix` (rows=anime) is transposed to `user_item_matrix` (rows=users) — `implicit`'s ALS expects user-item orientation, the opposite of what was built for item-item CF's similarity computation.

`factors=64` — the size of the learned vector per user/anime (an arbitrary but reasonable starting choice; larger captures more nuance at the cost of more compute). `iterations=20` — how many alternating solve-for-users/solve-for-anime rounds to run.

**Model is saved to disk (`als_model.pkl`) immediately after training** — this step is the most expensive in the whole pipeline at full scale, and the entire notebook was previously lost mid-session and had to be retrained from scratch once already.

In [16]:
os.environ['OPENBLAS_NUM_THREADS'] = '1'

item_user_matrix = load_npz(os.path.join(DATA_DIR, "item_user_matrix.npz"))

user_item_matrix = item_user_matrix.T.tocsr()

model = implicit.als.AlternatingLeastSquares(factors=64, regularization=0.01, iterations=20)
model.fit(user_item_matrix)

with open(os.path.join(DATA_DIR, "als_model.pkl"), "wb") as f:
    pickle.dump(model, f)

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

### Generating recommendations for one user

`model.recommend()` handles the ranking and exclusion of already-seen anime internally (via the `user_item_matrix[user_idx]` row passed in) — unlike item-item CF, no manual `-1` masking needed here. Single-user sanity check before full evaluation.

In [17]:
def als_recommend_for_user(user_idx, k=10):
    recommended = model.recommend(
        user_idx,
        user_item_matrix[user_idx],  # this user's row — used to exclude already-seen items
        N=k
    )
    item_indices, scores = recommended
    return [anime_id_map[i] for i in item_indices]

sample_uid = 5
recs = als_recommend_for_user(sample_uid, k=10)
print(animes[animes['animeID'].isin(recs)][['animeID', 'title']])



      animeID                                    title
11         12                             Cowboy Bebop
12         13                      Fullmetal Alchemist
37         38                         Samurai Champloo
47         48                            Dragon Ball Z
100       101                             Vinland Saga
147       148  Code Geass: Lelouch of the Rebellion R2
282       283                    Great Teacher Onizuka
408       409                                  Clannad
970       971                      Parasyte: The Maxim
1186     1187                            Steins;Gate 0


### Full evaluation

Same `precision_recall_*` pattern as the baseline notebook, swapped to call `als_recommend_for_user`.

**Result: Precision@10 = 0.1782, Recall@10 = 0.1740** — a modest but real improvement over item-item CF (0.1735 / 0.1645), consistent with ALS being a more principled version of the same underlying idea (behavioral similarity), rather than a fundamentally different signal.



In [18]:
def precision_recall_als(test_df, k=10, sample_users=None):
    total_relevant = 0
    total_recommended_relevant = 0

    test_positive = test_df[test_df['is_positive'] == 1]
    grouped = test_positive.groupby('user_idx')

    users_to_eval = list(grouped.groups.keys())
    if sample_users:
        users_to_eval = np.random.choice(users_to_eval, size=sample_users, replace=False)

    for user_idx in users_to_eval:
        group = grouped.get_group(user_idx)
        actual_positive = set(group['anime_id'])

        recommended = set(als_recommend_for_user(user_idx, k=k))

        hits_this_user = len(actual_positive & recommended)
        total_recommended_relevant += hits_this_user
        total_relevant += len(actual_positive)

    recall = total_recommended_relevant / total_relevant if total_relevant > 0 else 0
    precision = total_recommended_relevant / (len(users_to_eval) * k)

    return precision, recall

test_positive_full = test[test['is_positive'] == 1].dropna(subset=['anime_idx', 'user_idx'])
test_positive_full['anime_idx'] = test_positive_full['anime_idx'].astype(int)
test_positive_full['user_idx'] = test_positive_full['user_idx'].astype(int)

precision, recall = precision_recall_als(test_positive_full, k=10, sample_users=2000)
print(f"ALS — Precision@10: {precision:.4f}")
print(f"ALS — Recall@10: {recall:.4f}")

ALS — Precision@10: 0.1805
ALS — Recall@10: 0.1630


In [21]:
shutil.copytree('/content/media_recommender/data', '/content/drive/MyDrive/media_recommender_data', dirs_exist_ok=True)
print(os.listdir('/content/drive/MyDrive/media_recommender_data'))

['anime_data.jsonl', 'manga_data.jsonl', 'anilist_to_mal.json', 'train_ratings.parquet', 'test_ratings.parquet', 'manga_ratings.parquet', 'animes.csv.parquet', 'animes.parquet', 'item_user_matrix.npz', 'item_similarity.npz', 'als_model.pkl']
